11. El Costo del Marketing Inútil

In [0]:
%sql
SELECT 
    SUM(1.5) AS Fuga_Capital_Agencias
FROM cursodataacademy.proyecto_dataandino.hotel_bookings
WHERE distribution_channel = 'TA/TO' AND is_canceled = 1;

12. El Costo de Cancelar Tarde

In [0]:
%sql
WITH ReservasConFechas AS (
    SELECT 
        is_canceled,
        TO_DATE(CONCAT(arrival_date_year, '-', arrival_date_month, '-', arrival_date_day_of_month), 'yyyy-MMMM-d') AS arrival_date_real,
        TO_DATE(reservation_status_date, 'd/MM/yy') AS status_date_real
    FROM cursodataacademy.proyecto_dataandino.hotel_bookings
)
SELECT 
    COUNT(*) * 120 AS Costos_Hundidos_Cancelacion
FROM ReservasConFechas
WHERE is_canceled = 1 
AND DATEDIFF(arrival_date_real, status_date_real) < 3;

PREPARACIÓN PARA JOINS (Creación e inserción de tablas maestras)

In [0]:
%sql
-- Creación e inserción: Países
CREATE TABLE IF NOT EXISTS cursodataacademy.proyecto_dataandino.tb_paises (
    cod_pais STRING,
    nombre_pais STRING
);
INSERT INTO cursodataacademy.proyecto_dataandino.tb_paises VALUES 
('PRT', 'Portugal'), ('GBR', 'Reino Unido'), ('FRA', 'Francia'), 
('ESP', 'España'), ('DEU', 'Alemania');

-- Creación e inserción: Vendedores (ID adaptado a la columna agent)
CREATE TABLE IF NOT EXISTS cursodataacademy.proyecto_dataandino.tb_vendedores (
    id_vendedor DOUBLE,
    nombre_vendedor STRING
);
INSERT INTO cursodataacademy.proyecto_dataandino.tb_vendedores VALUES 
(9.0, 'Carlos Perez'), (240.0, 'Maria Gomez'), (14.0, 'Luis Torres');

-- Creación e inserción: Facturas
CREATE TABLE IF NOT EXISTS cursodataacademy.proyecto_dataandino.tb_facturas (
    id_reserva BIGINT,
    monto_facturado DECIMAL(10,2)
);
INSERT INTO cursodataacademy.proyecto_dataandino.tb_facturas VALUES 
(1, 150.00), (2, 300.50), (3, 450.00);

13. (JOIN 1 - LEFT JOIN)

In [0]:
%sql
SELECT 
    res.is_canceled, 
    pais.nombre_pais
FROM cursodataacademy.proyecto_dataandino.hotel_bookings res
LEFT JOIN cursodataacademy.proyecto_dataandino.tb_paises pais 
    ON res.country = pais.cod_pais;

14. (JOIN 2 - INNER JOIN con Agrupación)

In [0]:
%sql
SELECT 
    pais.nombre_pais,
    COUNT(*) AS Total_Canceladas
FROM cursodataacademy.proyecto_dataandino.hotel_bookings res
INNER JOIN cursodataacademy.proyecto_dataandino.tb_paises pais 
    ON res.country = pais.cod_pais
WHERE res.is_canceled = 1
GROUP BY pais.nombre_pais
ORDER BY Total_Canceladas DESC
LIMIT 5;

15. (JOIN 3 - FULL OUTER JOIN)

In [0]:
%sql
SELECT 
    res.distribution_channel,
    vend.nombre_vendedor
FROM cursodataacademy.proyecto_dataandino.hotel_bookings res
FULL OUTER JOIN cursodataacademy.proyecto_dataandino.tb_vendedores vend 
    ON res.agent = vend.id_vendedor;

16. (JOIN 4 - RIGHT JOIN)

In [0]:
%sql
WITH ReservasConID AS (
    SELECT 
        monotonically_increasing_id() AS id_reserva,
        hotel
    FROM cursodataacademy.proyecto_dataandino.hotel_bookings
)
SELECT 
    fac.id_reserva,
    fac.monto_facturado,
    res.hotel
FROM ReservasConID res
RIGHT JOIN cursodataacademy.proyecto_dataandino.tb_facturas fac 
    ON res.id_reserva = fac.id_reserva;